# Monthly Skill Metrics by Company AI Strategy

Monthly-granularity version of `company_bucket_strategy_analysis.ipynb`.

**Outputs** (all use snake_case column names compatible with DGM.R):

1. `monthly_bucket_metrics.csv` — one row per (classification, month), with 4 metrics × 6 buckets = 24 metric columns
2. `monthly_category_metrics.csv` — same, but 4 × 133 = 532 metric columns
3. `rcid_monthly_buckets/{rcid}_{name}.csv` — one file per classified company; rows are months
4. `rcid_monthly_categories/{rcid}_{name}.csv` — same, but at category granularity

**4 metrics per group per month:**
- `*_pct_posts`              — % of posts mentioning ≥1 skill in this bucket/category
- `*_n_posts`                — absolute count of posts mentioning ≥1 skill
- `*_total_mentions`         — sum of all skill mentions (a post can contribute multiple)
- `*_avg_mentions_per_post`  — total_mentions / total_posts (monthly denominator)


In [ ]:
# =============================================================================
# CONFIGURATION & IMPORTS
# =============================================================================

import pandas as pd
import numpy as np
import os
import re
import ast
import gc
import glob
from pathlib import Path
from collections import Counter, defaultdict
from typing import List

INPUT_FOLDER        = '../data/extracted_skills_production'
SKILLS_COLUMN       = 'skills'
DATE_COLUMN         = 'post_date'
RCID_COLUMN         = 'rcid'
CLASSIFICATION_FILE = '500.xlsx'
BUCKET_MAPPING_FILE = '../skillner/data/ksao_full_mapping_with_buckets.csv'

# --- Alternate implementation-time source (overrides 500.xlsx time column) ---
# Set IMPL_TIME_FILE=None to fall back to the 500.xlsx `Implementation time` column.
IMPL_TIME_FILE    = 'merged_ai_implementation_times.xlsx'
IMPL_TIME_COL     = 'First AI Post Date (Source 2: Job Posts)'
IMPL_TIME_NAMECOL = 'Company (File 1 name)'   # joins to 500.xlsx `Company name`
DGM_OUTPUT_SUFFIX = '_jobposts'               # appended to dgm_monthly_* output filenames

OUT_DIR_BUCKET_RCID   = 'rcid_monthly_buckets'
OUT_DIR_CATEGORY_RCID = 'rcid_monthly_categories'
OUT_FILE_BUCKET_AGG   = 'monthly_bucket_metrics.csv'
OUT_FILE_CATEGORY_AGG = 'monthly_category_metrics.csv'


def parse_skills_column(value) -> List[str]:
    if value is None:
        return []
    if isinstance(value, np.ndarray):
        return [str(s) for s in value if s is not None]
    if isinstance(value, str):
        if value in ('[]', '', 'nan', 'None'):
            return []
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list):
                return parsed
        except (ValueError, SyntaxError):
            pass
    return []


def snake(s: str) -> str:
    s = str(s).lower()
    s = s.replace('&', 'and')
    s = re.sub(r'[^a-z0-9]+', '_', s).strip('_')
    return s


METRIC_SUFFIXES = ('pct_posts', 'n_posts', 'total_mentions', 'avg_mentions_per_post')


# --- Load skill → bucket / category lookups ---
bucket_df = pd.read_csv(BUCKET_MAPPING_FILE)
_skill_key = bucket_df['skill_name'].str.lower().str.strip()
BUCKET_LOOKUP   = dict(zip(_skill_key, bucket_df['broad_bucket']))
CATEGORY_LOOKUP = dict(zip(_skill_key, bucket_df['category']))

ALL_BUCKETS    = sorted(bucket_df['broad_bucket'].dropna().unique().tolist())
ALL_CATEGORIES = sorted(bucket_df['category'].dropna().unique().tolist())

BUCKET_COL_PREFIX = {b: f'bucket_{snake(b)}' for b in ALL_BUCKETS}
CAT_COL_PREFIX    = {c: f'cat_{snake(c)}'    for c in ALL_CATEGORIES}

# Collision guards — if normalized names collide, downstream CSVs would fuse columns
assert len(set(BUCKET_COL_PREFIX.values())) == len(ALL_BUCKETS), "bucket name collision"
assert len(set(CAT_COL_PREFIX.values()))    == len(ALL_CATEGORIES), "category name collision"

print(f"Loaded {len(BUCKET_LOOKUP):,} skill mappings")
print(f"Broad buckets ({len(ALL_BUCKETS)}): {ALL_BUCKETS}")
print(f"Categories: {len(ALL_CATEGORIES)} unique values")

parquet_files = sorted(glob.glob(f'{INPUT_FOLDER}/*.parquet'))
print(f"Found {len(parquet_files)} parquet files")


In [ ]:
# =============================================================================
# LOAD COMPANY CLASSIFICATIONS (500.xlsx)
# =============================================================================

import openpyxl

wb = openpyxl.load_workbook(CLASSIFICATION_FILE, read_only=True, data_only=True)
ws = wb['Sheet1']

headers = [cell.value for cell in next(ws.iter_rows(min_row=1, max_row=1))]
print("Columns:", headers)

rcid_col  = headers.index('rcid')
class_col = headers.index('Raisch & Krakowski (2021) Classification')
name_col  = headers.index('Company name')

VALID_CLASSES = {'Augmentation', 'Automation', 'Both'}

company_rows = []
for row in ws.iter_rows(min_row=2, values_only=True):
    rcid_val = row[rcid_col]
    classification = row[class_col]
    company_name = row[name_col]
    if rcid_val is None or str(rcid_val).strip() in ('#N/A', '', 'N/A', 'nan'):
        continue
    if classification not in VALID_CLASSES:
        continue
    try:
        rcid_int = int(float(str(rcid_val)))
    except (ValueError, TypeError):
        continue
    company_rows.append((rcid_int, company_name, classification))

wb.close()

rcid_classes = defaultdict(set)
rcid_names = {}
for rcid_int, company_name, classification in company_rows:
    rcid_classes[rcid_int].add(classification)
    rcid_names[rcid_int] = company_name

rcid_to_category = {}
for rcid_int, classes in rcid_classes.items():
    if 'Both' in classes or {'Augmentation', 'Automation'}.issubset(classes):
        rcid_to_category[rcid_int] = 'Both'
    elif 'Automation' in classes:
        rcid_to_category[rcid_int] = 'Automation'
    else:
        rcid_to_category[rcid_int] = 'Augmentation'

rcid_to_category_str = {str(k): v for k, v in rcid_to_category.items()}

category_counts = Counter(rcid_to_category.values())
print(f"\nCompany classifications (by unique RCID):")
print(f"  Total classified: {len(rcid_to_category)}")
for cat in ['Augmentation', 'Automation', 'Both']:
    print(f"  {cat}: {category_counts.get(cat, 0)}")

CLASSES = ['Augmentation', 'Automation', 'Both', 'Other']


In [ ]:
# =============================================================================
# STREAM PARQUET FILES: Accumulate per-classification + per-RCID monthly metrics
# =============================================================================

def _zero_bucket_d(): return {b: 0 for b in ALL_BUCKETS}
def _zero_cat_d():    return {c: 0 for c in ALL_CATEGORIES}

# --- Per-classification × month ---
cls_monthly_posts         = {cls: defaultdict(int)              for cls in CLASSES}
cls_monthly_bucket_npost  = {cls: defaultdict(_zero_bucket_d)   for cls in CLASSES}
cls_monthly_bucket_ments  = {cls: defaultdict(_zero_bucket_d)   for cls in CLASSES}
cls_monthly_cat_npost     = {cls: defaultdict(_zero_cat_d)      for cls in CLASSES}
cls_monthly_cat_ments     = {cls: defaultdict(_zero_cat_d)      for cls in CLASSES}

# --- Per-RCID × month (classified companies only) ---
rcid_monthly_posts         = defaultdict(lambda: defaultdict(int))
rcid_monthly_bucket_npost  = defaultdict(lambda: defaultdict(_zero_bucket_d))
rcid_monthly_bucket_ments  = defaultdict(lambda: defaultdict(_zero_bucket_d))
rcid_monthly_cat_npost     = defaultdict(lambda: defaultdict(_zero_cat_d))
rcid_monthly_cat_ments     = defaultdict(lambda: defaultdict(_zero_cat_d))

print(f"Streaming {len(parquet_files)} parquet files...")

for i, filepath in enumerate(parquet_files):
    try:
        df_part = pd.read_parquet(filepath, columns=[RCID_COLUMN, DATE_COLUMN, SKILLS_COLUMN])
    except Exception as e:
        print(f"  Skipping {filepath}: {e}")
        continue

    df_part[DATE_COLUMN] = pd.to_datetime(df_part[DATE_COLUMN], errors='coerce')
    df_part['month_str'] = df_part[DATE_COLUMN].dt.strftime('%Y-%m')

    for _, row in df_part.iterrows():
        m = row['month_str']
        if pd.isna(m) or m == 'NaT':
            continue

        rcid_val = row[RCID_COLUMN]
        rcid_str = str(int(float(rcid_val))) if pd.notna(rcid_val) else ''
        cls = rcid_to_category_str.get(rcid_str, 'Other')
        is_classified = rcid_str in rcid_to_category_str

        # Denominator: every post contributes to total_posts (even with empty skills)
        cls_monthly_posts[cls][m] += 1
        if is_classified:
            rcid_monthly_posts[rcid_str][m] += 1

        skills_list = parse_skills_column(row.get(SKILLS_COLUMN))

        seen_buckets, seen_cats = set(), set()
        for skill in skills_list:
            s = skill.lower().strip()
            if s in BUCKET_LOOKUP:
                b = BUCKET_LOOKUP[s]
                cls_monthly_bucket_ments[cls][m][b] += 1
                if is_classified:
                    rcid_monthly_bucket_ments[rcid_str][m][b] += 1
                seen_buckets.add(b)
            if s in CATEGORY_LOOKUP:
                c = CATEGORY_LOOKUP[s]
                cls_monthly_cat_ments[cls][m][c] += 1
                if is_classified:
                    rcid_monthly_cat_ments[rcid_str][m][c] += 1
                seen_cats.add(c)

        for b in seen_buckets:
            cls_monthly_bucket_npost[cls][m][b] += 1
            if is_classified:
                rcid_monthly_bucket_npost[rcid_str][m][b] += 1
        for c in seen_cats:
            cls_monthly_cat_npost[cls][m][c] += 1
            if is_classified:
                rcid_monthly_cat_npost[rcid_str][m][c] += 1

    del df_part
    gc.collect()
    if (i + 1) % 10 == 0:
        print(f"  [{i+1}/{len(parquet_files)}] processed")

print("\nDone streaming.")
print("Classification totals:")
for cls in CLASSES:
    n_months = len(cls_monthly_posts[cls])
    n_posts  = sum(cls_monthly_posts[cls].values())
    print(f"  {cls:15s} {n_months:3d} months, {n_posts:>10,} posts")

print(f"Per-RCID data collected for {len(rcid_monthly_posts)} classified companies")


In [ ]:
# =============================================================================
# AGGREGATE: monthly_bucket_metrics.csv (4 metrics × 6 buckets, per classification)
# =============================================================================

def build_monthly_rows(posts_by_month, npost_by_month, ments_by_month,
                       groups, group_prefix):
    """Build a list of dict rows, one per month that has >=1 post."""
    rows = []
    for m in sorted(posts_by_month.keys()):
        tp = posts_by_month[m]
        if tp == 0:
            continue
        row = {'month': m, 'total_posts': tp}
        for g in groups:
            prefix = group_prefix[g]
            n = npost_by_month[m][g]
            t = ments_by_month[m][g]
            row[f'{prefix}_pct_posts']             = 100.0 * n / tp
            row[f'{prefix}_n_posts']               = n
            row[f'{prefix}_total_mentions']        = t
            row[f'{prefix}_avg_mentions_per_post'] = t / tp
        rows.append(row)
    return rows


agg_bucket_parts = []
for cls in CLASSES:
    rows = build_monthly_rows(
        cls_monthly_posts[cls],
        cls_monthly_bucket_npost[cls],
        cls_monthly_bucket_ments[cls],
        ALL_BUCKETS, BUCKET_COL_PREFIX,
    )
    if rows:
        df_cls = pd.DataFrame(rows)
        df_cls.insert(0, 'classification', cls)
        agg_bucket_parts.append(df_cls)

if agg_bucket_parts:
    monthly_bucket_agg = pd.concat(agg_bucket_parts, ignore_index=True)
    monthly_bucket_agg.to_csv(OUT_FILE_BUCKET_AGG, index=False)
    print(f"Saved: {OUT_FILE_BUCKET_AGG} "
          f"({len(monthly_bucket_agg)} rows, {monthly_bucket_agg.shape[1]} columns)")
    preview_cols = ['classification', 'month', 'total_posts'] + \
        [c for c in monthly_bucket_agg.columns if c.startswith(BUCKET_COL_PREFIX[ALL_BUCKETS[0]] + '_')]
    print(monthly_bucket_agg[preview_cols].head(10).to_string(index=False))
else:
    print("No aggregate bucket data.")


In [ ]:
# =============================================================================
# AGGREGATE: monthly_category_metrics.csv (4 metrics × 133 categories)
# =============================================================================

agg_cat_parts = []
for cls in CLASSES:
    rows = build_monthly_rows(
        cls_monthly_posts[cls],
        cls_monthly_cat_npost[cls],
        cls_monthly_cat_ments[cls],
        ALL_CATEGORIES, CAT_COL_PREFIX,
    )
    if rows:
        df_cls = pd.DataFrame(rows)
        df_cls.insert(0, 'classification', cls)
        agg_cat_parts.append(df_cls)

if agg_cat_parts:
    monthly_cat_agg = pd.concat(agg_cat_parts, ignore_index=True)
    monthly_cat_agg.to_csv(OUT_FILE_CATEGORY_AGG, index=False)
    print(f"Saved: {OUT_FILE_CATEGORY_AGG} "
          f"({len(monthly_cat_agg)} rows, {monthly_cat_agg.shape[1]} columns)")
    preview_cols = ['classification', 'month', 'total_posts'] + \
        [c for c in monthly_cat_agg.columns if c.startswith(CAT_COL_PREFIX[ALL_CATEGORIES[0]] + '_')]
    print(monthly_cat_agg[preview_cols].head(10).to_string(index=False))
else:
    print("No aggregate category data.")


In [ ]:
# =============================================================================
# PER-RCID MONTHLY BUCKET CSVs → rcid_monthly_buckets/
# =============================================================================

os.makedirs(OUT_DIR_BUCKET_RCID, exist_ok=True)

exported = 0
for rcid_str, posts_by_month in rcid_monthly_posts.items():
    rows = build_monthly_rows(
        posts_by_month,
        rcid_monthly_bucket_npost[rcid_str],
        rcid_monthly_bucket_ments[rcid_str],
        ALL_BUCKETS, BUCKET_COL_PREFIX,
    )
    if not rows:
        continue

    rcid_int       = int(rcid_str)
    company_name   = rcid_names.get(rcid_int, 'unknown')
    category_label = rcid_to_category.get(rcid_int, 'Unknown')

    safe_name = re.sub(r'[^\w\s-]', '', str(company_name)).strip().replace(' ', '_')
    filename  = f'{rcid_str}_{safe_name}.csv'

    df_rcid = pd.DataFrame(rows)
    df_rcid.insert(0, 'rcid', rcid_str)
    df_rcid.insert(1, 'company_name', company_name)
    df_rcid.insert(2, 'category', category_label)
    df_rcid.to_csv(os.path.join(OUT_DIR_BUCKET_RCID, filename), index=False)
    exported += 1

print(f"Exported {exported} per-RCID monthly bucket CSVs → {OUT_DIR_BUCKET_RCID}/")

# Helper: column-name list for downstream R/Python use
bucket_col_names = [
    f'{BUCKET_COL_PREFIX[b]}_{suf}'
    for b in ALL_BUCKETS for suf in METRIC_SUFFIXES
]
pd.Series(bucket_col_names, name='outcome_column').to_csv(
    os.path.join(OUT_DIR_BUCKET_RCID, '_bucket_columns.csv'), index=False)
print(f"Wrote column-name helper: {OUT_DIR_BUCKET_RCID}/_bucket_columns.csv "
      f"({len(bucket_col_names)} entries)")


In [ ]:
# =============================================================================
# PER-RCID MONTHLY CATEGORY CSVs → rcid_monthly_categories/
# =============================================================================

os.makedirs(OUT_DIR_CATEGORY_RCID, exist_ok=True)

exported = 0
for rcid_str, posts_by_month in rcid_monthly_posts.items():
    rows = build_monthly_rows(
        posts_by_month,
        rcid_monthly_cat_npost[rcid_str],
        rcid_monthly_cat_ments[rcid_str],
        ALL_CATEGORIES, CAT_COL_PREFIX,
    )
    if not rows:
        continue

    rcid_int       = int(rcid_str)
    company_name   = rcid_names.get(rcid_int, 'unknown')
    category_label = rcid_to_category.get(rcid_int, 'Unknown')

    safe_name = re.sub(r'[^\w\s-]', '', str(company_name)).strip().replace(' ', '_')
    filename  = f'{rcid_str}_{safe_name}.csv'

    df_rcid = pd.DataFrame(rows)
    df_rcid.insert(0, 'rcid', rcid_str)
    df_rcid.insert(1, 'company_name', company_name)
    df_rcid.insert(2, 'category', category_label)
    df_rcid.to_csv(os.path.join(OUT_DIR_CATEGORY_RCID, filename), index=False)
    exported += 1

print(f"Exported {exported} per-RCID monthly category CSVs → {OUT_DIR_CATEGORY_RCID}/")

category_col_names = [
    f'{CAT_COL_PREFIX[c]}_{suf}'
    for c in ALL_CATEGORIES for suf in METRIC_SUFFIXES
]
pd.Series(category_col_names, name='outcome_column').to_csv(
    os.path.join(OUT_DIR_CATEGORY_RCID, '_category_columns.csv'), index=False)
print(f"Wrote column-name helper: {OUT_DIR_CATEGORY_RCID}/_category_columns.csv "
      f"({len(category_col_names)} entries)")


In [ ]:
# =============================================================================
# DGM PREP — parse implementation dates, build panel, fit helper (statsmodels)
# =============================================================================
# Port of DGM code.R (Discontinuous Growth Modeling) to Python.
# Runs lme(DV ~ TIME + TRANS + RECOV, random = ~1|rcid) via statsmodels.MixedLM
# on each outcome × {Augmentation, Automation, Both} pair.

import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings('ignore')

from openpyxl import load_workbook


def _parse_first_date(x):
    """Return (year:int, month:int) or (None, None).
    Accepts: datetime, Excel serial number, ISO string (YYYY-MM-DD or YYYY/MM/DD),
    M/YYYY, Q#/YYYY, free-text with a year."""
    if x is None:
        return (None, None)

    # datetime / date objects (openpyxl returns these when the cell is date-formatted)
    import datetime as _dt
    if isinstance(x, (_dt.datetime, _dt.date)):
        return (x.year, x.month)

    # Excel serial number (e.g. 43964 → 2020-05-08)
    if isinstance(x, (int, float)) and not isinstance(x, bool):
        try:
            d = _dt.datetime(1899, 12, 30) + _dt.timedelta(days=int(x))
            return (d.year, d.month)
        except (OverflowError, ValueError):
            return (None, None)

    s = str(x).strip()
    if s in ('', 'NA', 'nan', 'None', '#N/A'):
        return (None, None)

    # ISO-style: YYYY-MM-DD or YYYY/MM/DD
    m = re.match(r'^(\d{4})[-/](\d{1,2})(?:[-/]\d{1,2})?$', s)
    if m:
        return (int(m.group(1)), int(m.group(2)))

    m = re.search(r'\b(0?[1-9]|1[0-2])/(\d{4})\b', s)
    if m:
        return (int(m.group(2)), int(m.group(1)))

    m = re.search(r'Q([1-4])\s*(\d{4})', s)
    if m:
        return (int(m.group(2)), (int(m.group(1)) - 1) * 3 + 1)

    low = s.lower()
    yr_match = re.search(r'\d{4}', s)
    if 'summer' in low and yr_match:
        return (int(yr_match.group()), 7)
    if 'early' in low and yr_match:
        return (int(yr_match.group()), 1)
    if yr_match:
        return (int(yr_match.group()), 1)
    return (None, None)


# --- Build rcid ↔ (name, classification) map from 500.xlsx ---
# We still need 500.xlsx for rcid + Raisch & Krakowski classification;
# only the *time* comes from IMPL_TIME_FILE (if set).
wb = load_workbook(CLASSIFICATION_FILE, read_only=True, data_only=True)
ws = wb['Sheet1']
headers = [c.value for c in next(ws.iter_rows(min_row=1, max_row=1))]
_idx = {h: i for i, h in enumerate(headers)}
RCID_IDX  = _idx['rcid']
CLASS_IDX = _idx['Raisch & Krakowski (2021) Classification']
NAME_IDX  = _idx['Company name']
TIME_IDX  = _idx['Implementation time (ideally specify to date or month)']

CLS_SHORT = {'Automation': 'auto', 'Augmentation': 'aug', 'Both': 'both'}


def _norm_company(name):
    """Normalize company names for cross-file matching."""
    if name is None:
        return ''
    s = str(name).strip().casefold()
    # Strip trailing legal suffixes / punctuation
    s = re.sub(r'[.,]', '', s)
    s = re.sub(
        r'\s+(inc|incorporated|corp|corporation|co|company|ltd|limited|lp|llc|plc|holdings?|group|sa|ag|nv|se)$',
        '', s,
    ).strip()
    s = re.sub(r'\s+', ' ', s)
    return s


# name → [(rcid_int, classification_short), ...]
name_to_rcid_cls = defaultdict(list)
# rcid → {short: 'YYYY-MM'} with 500.xlsx times (used only as fallback)
_fallback_rcid_trans_month = defaultdict(dict)

for row in ws.iter_rows(min_row=2, values_only=True):
    rcid_val = row[RCID_IDX]
    cls      = row[CLASS_IDX]
    name     = row[NAME_IDX]
    tval     = row[TIME_IDX]
    if rcid_val is None or cls not in CLS_SHORT:
        continue
    try:
        rcid_int = int(float(str(rcid_val)))
    except (ValueError, TypeError):
        continue
    short = CLS_SHORT[cls]
    key = _norm_company(name)
    if key:
        name_to_rcid_cls[key].append((rcid_int, short))
    yr, mo = _parse_first_date(tval)
    if yr is not None:
        new_key = yr * 100 + mo
        existing = _fallback_rcid_trans_month[rcid_int].get(short)
        if existing is None or new_key < int(existing[:4]) * 100 + int(existing[5:7]):
            _fallback_rcid_trans_month[rcid_int][short] = f"{yr:04d}-{mo:02d}"

wb.close()

rcid_trans_month = defaultdict(dict)   # {rcid_int: {auto: 'YYYY-MM', ...}}

if IMPL_TIME_FILE:
    wb2 = load_workbook(IMPL_TIME_FILE, read_only=True, data_only=True)
    ws2 = wb2.active
    h2 = [c.value for c in next(ws2.iter_rows(min_row=1, max_row=1))]
    try:
        time_idx = h2.index(IMPL_TIME_COL)
        name_idx = h2.index(IMPL_TIME_NAMECOL)
    except ValueError as e:
        wb2.close()
        raise ValueError(f"{IMPL_TIME_FILE} missing expected column: {e}")

    matched_companies = 0
    unmatched_names = []
    rows_with_time = 0
    for row in ws2.iter_rows(min_row=2, values_only=True):
        name = row[name_idx]
        tval = row[time_idx]
        if name is None:
            continue
        yr, mo = _parse_first_date(tval)
        if yr is None:
            continue
        rows_with_time += 1
        key = _norm_company(name)
        pairs = name_to_rcid_cls.get(key, [])
        if not pairs:
            if len(unmatched_names) < 20:
                unmatched_names.append(str(name))
            continue
        matched_companies += 1
        ym = f"{yr:04d}-{mo:02d}"
        for rcid_int, short in pairs:
            existing = rcid_trans_month[rcid_int].get(short)
            new_key = yr * 100 + mo
            if existing is None or new_key < int(existing[:4]) * 100 + int(existing[5:7]):
                rcid_trans_month[rcid_int][short] = ym
    wb2.close()

    print(f"[IMPL_TIME] Using time column '{IMPL_TIME_COL}' from {IMPL_TIME_FILE}")
    print(f"[IMPL_TIME] Rows with a parseable date: {rows_with_time}")
    print(f"[IMPL_TIME] Matched to 500.xlsx companies: {matched_companies}")
    if unmatched_names:
        print(f"[IMPL_TIME] Unmatched (first {len(unmatched_names)}):")
        for n in unmatched_names:
            print(f"    {n!r}")
else:
    rcid_trans_month = _fallback_rcid_trans_month

print(f"Parsed impl_time for {len(rcid_trans_month)} rcids")
print(f"  with auto: {sum('auto' in v for v in rcid_trans_month.values())}")
print(f"  with aug : {sum('aug'  in v for v in rcid_trans_month.values())}")
print(f"  with both: {sum('both' in v for v in rcid_trans_month.values())}")


def _build_panel(folder):
    """Read per-RCID CSVs, add TIME / TRANS_* / RECOV_* columns."""
    csvs = sorted(
        f for f in glob.glob(f'{folder}/*.csv')
        if not os.path.basename(f).startswith('_')
           and os.path.basename(f) != 'combined_dataset.csv'
    )
    parts = []
    for fp in csvs:
        df = pd.read_csv(fp)
        if df.empty or 'month' not in df.columns:
            continue
        rcid_int = int(df['rcid'].iloc[0])
        df = df.sort_values('month').reset_index(drop=True)
        df['TIME'] = np.arange(len(df), dtype=int)
        months = df['month'].tolist()
        for short in ('auto', 'aug', 'both'):
            tm = rcid_trans_month.get(rcid_int, {}).get(short)
            if tm is None or tm not in months:
                df[f'TRANS_{short}'] = np.nan
                df[f'RECOV_{short}'] = np.nan
            else:
                idx = months.index(tm)
                df[f'TRANS_{short}'] = np.where(df['TIME'] >= idx, 1, 0)
                df[f'RECOV_{short}'] = np.where(df['TIME'] >= idx, df['TIME'] - idx, 0)
        parts.append(df)
    if not parts:
        return pd.DataFrame()
    return pd.concat(parts, ignore_index=True)


def _fit_one(dat):
    """Fit ICC + step1a (DV ~ TIME + TRANS + RECOV, random = ~1|rcid).
    Return (icc, coef_rows) or (icc, []) if step1a fails."""
    icc = np.nan
    try:
        s1 = smf.mixedlm('DV ~ 1', data=dat, groups=dat['rcid']).fit(
            reml=True, method='lbfgs')
        tau   = float(s1.cov_re.iloc[0, 0])
        sigma = float(s1.scale)
        denom = tau + sigma
        if denom > 0:
            icc = round(tau / denom, 3)
    except Exception:
        pass

    try:
        m = smf.mixedlm('DV ~ TIME + TRANS + RECOV', data=dat, groups=dat['rcid']).fit(
            reml=True, method='lbfgs')
        rows = []
        for term in m.params.index:
            rows.append({
                'term':     term,
                'Value':    float(m.params[term]),
                'Std.Error': float(m.bse[term]),
                't-value':  float(m.tvalues[term]),
                'p-value':  float(m.pvalues[term]),
            })
        return icc, rows
    except Exception as e:
        return icc, [{'term': f'ERROR: {type(e).__name__}', 'Value': np.nan,
                      'Std.Error': np.nan, 't-value': np.nan, 'p-value': np.nan}]


def run_dgm(folder, metric_suffix, output_csv, min_rows=10, verbose_every=25):
    """Full DGM pipeline for a folder + single metric suffix."""
    helper = glob.glob(f'{folder}/_*_columns.csv')
    if not helper:
        print(f"[run_dgm] No helper CSV in {folder}; skipping.")
        return None
    outcomes = pd.read_csv(helper[0])['outcome_column'].tolist()
    if metric_suffix:
        outcomes = [o for o in outcomes if o.endswith(f'_{metric_suffix}')]

    panel = _build_panel(folder)
    if panel.empty:
        print(f"[run_dgm] Panel is empty for {folder}")
        return None
    outcomes = [o for o in outcomes if o in panel.columns]
    classifications = ['aug', 'auto', 'both']

    print(f"\n→ {folder} / {metric_suffix or 'ALL'}: "
          f"{len(outcomes)} outcomes × {len(classifications)} classes "
          f"= {len(outcomes) * len(classifications)} models")

    all_rows = []
    total = len(outcomes) * len(classifications)
    done = 0
    for outcome in outcomes:
        for cls in classifications:
            done += 1
            TRANS_col, RECOV_col = f'TRANS_{cls}', f'RECOV_{cls}'
            sub = panel[['rcid', 'month', 'TIME', TRANS_col, RECOV_col, outcome]].copy()
            sub.columns = ['rcid', 'month', 'TIME', 'TRANS', 'RECOV', 'DV']
            sub = sub.dropna(subset=['DV', 'TIME', 'TRANS'])
            if len(sub) < min_rows or sub['TRANS'].nunique() < 2:
                continue
            icc, rows = _fit_one(sub)
            for r in rows:
                r['outcome']        = outcome
                r['classification'] = cls
                r['icc']            = icc
                all_rows.append(r)
            if verbose_every and done % verbose_every == 0:
                print(f"    [{done}/{total}] {outcome} × {cls}")

    result = pd.DataFrame(all_rows)
    if result.empty:
        print(f"[run_dgm] No results for {output_csv}")
        return None
    result = result[['outcome', 'classification', 'icc', 'term',
                     'Value', 'Std.Error', 't-value', 'p-value']]
    result.to_csv(output_csv, index=False)
    print(f"  ✓ saved {len(result)} rows → {output_csv}")
    return result


print("DGM helpers ready.")


In [ ]:
# =============================================================================
# RUN DGM for all 4 metrics × {buckets, categories} → 8 CSV files
# =============================================================================
# Output files (in current working directory):
#   dgm_monthly_buckets_pct_posts.csv            (6 outcomes)
#   dgm_monthly_buckets_n_posts.csv
#   dgm_monthly_buckets_total_mentions.csv
#   dgm_monthly_buckets_avg_mentions_per_post.csv
#   dgm_monthly_categories_pct_posts.csv         (133 outcomes)
#   dgm_monthly_categories_n_posts.csv
#   dgm_monthly_categories_total_mentions.csv
#   dgm_monthly_categories_avg_mentions_per_post.csv

DGM_TARGETS = [
    (OUT_DIR_BUCKET_RCID,   'buckets'),
    (OUT_DIR_CATEGORY_RCID, 'categories'),
]

dgm_results = {}
for folder, scope in DGM_TARGETS:
    for suf in METRIC_SUFFIXES:
        out_csv = f'dgm_monthly_{scope}_{suf}{DGM_OUTPUT_SUFFIX}.csv'
        dgm_results[(scope, suf)] = run_dgm(folder, suf, out_csv)

print("\n=== DGM RUN COMPLETE ===")
for (scope, suf), df in dgm_results.items():
    n = 0 if df is None else len(df)
    print(f"  {scope:10s} / {suf:28s} → {n:>6,} coef rows")
